# 04 — Train an LSTM on the expanded UCF101 pose dataset

This notebook trains the same kind of pose-sequence LSTM as the 5-class baseline, but it reads the generalized dataset in `data/processed/sequences_all/`.

## Memory-safe design
- Loads `X_train.npy`, `X_val.npy`, and `X_test.npy` as **read-only memory maps** rather than copying all sequence data into RAM.
- Uses a PyTorch `Dataset` and minibatches.
- Uses GPU if CUDA is available; otherwise CPU.
- Saves the best checkpoint and training history after each epoch.
- Selects the checkpoint using validation macro-F1, then evaluates the held-out test set once.

Run the cells from top to bottom. First make sure notebook `03_batch_extract_all_ucf101_and_build_sequences.ipynb` finished and produced non-empty train/validation/test arrays.

In [1]:
from pathlib import Path
import json
import random
import copy
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
SEQUENCE_DIR = PROJECT_DIR / "data" / "processed" / "sequences_all"
OUTPUT_MODEL_DIR = PROJECT_DIR / "outputs" / "models"
OUTPUT_REPORT_DIR = PROJECT_DIR / "outputs" / "reports"
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Sequence data:", SEQUENCE_DIR)
print("Sequence folder exists:", SEQUENCE_DIR.is_dir())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert SEQUENCE_DIR.is_dir(), f"Sequence folder not found: {SEQUENCE_DIR}"

Project: C:\Users\admin\ViTPose_Action_Recognition
Sequence data: C:\Users\admin\ViTPose_Action_Recognition\data\processed\sequences_all
Sequence folder exists: True
PyTorch: 2.7.1+cu118
CUDA available: True
GPU: NVIDIA RTX A4000


In [2]:
# Reproducibility and memory-mapped loading
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
required_files = [
    "X_train.npy", "y_train.npy", "X_val.npy", "y_val.npy",
    "X_test.npy", "y_test.npy", "videos_train.npy", "videos_val.npy",
    "videos_test.npy", "label_map.json"
]
missing = [name for name in required_files if not (SEQUENCE_DIR / name).is_file()]
assert not missing, f"Missing files: {missing}. Finish the sequence-building notebook first."

# mmap_mode='r' keeps large feature arrays on disk and reads samples on demand.
X_train = np.load(SEQUENCE_DIR / "X_train.npy", mmap_mode="r", allow_pickle=False)
y_train = np.load(SEQUENCE_DIR / "y_train.npy", mmap_mode="r", allow_pickle=False)
X_val = np.load(SEQUENCE_DIR / "X_val.npy", mmap_mode="r", allow_pickle=False)
y_val = np.load(SEQUENCE_DIR / "y_val.npy", mmap_mode="r", allow_pickle=False)
X_test = np.load(SEQUENCE_DIR / "X_test.npy", mmap_mode="r", allow_pickle=False)
y_test = np.load(SEQUENCE_DIR / "y_test.npy", mmap_mode="r", allow_pickle=False)
videos_train = np.load(SEQUENCE_DIR / "videos_train.npy", mmap_mode="r", allow_pickle=False)
videos_val = np.load(SEQUENCE_DIR / "videos_val.npy", mmap_mode="r", allow_pickle=False)
videos_test = np.load(SEQUENCE_DIR / "videos_test.npy", mmap_mode="r", allow_pickle=False)

label_map = json.loads((SEQUENCE_DIR / "label_map.json").read_text(encoding="utf-8"))
id_to_label = {int(k): str(v) for k, v in label_map.items()}
class_ids = sorted(id_to_label)
class_names = [id_to_label[i] for i in class_ids]
NUM_CLASSES = len(class_names)

print("Device:", DEVICE)
print("Classes:", NUM_CLASSES)
print("Label map:", id_to_label)
print("Train:", X_train.shape, y_train.shape, "| videos:", len(np.unique(videos_train)))
print("Validation:", X_val.shape, y_val.shape, "| videos:", len(np.unique(videos_val)))
print("Test:", X_test.shape, y_test.shape, "| videos:", len(np.unique(videos_test)))

AssertionError: Missing files: ['X_train.npy', 'y_train.npy', 'X_val.npy', 'y_val.npy', 'X_test.npy', 'y_test.npy', 'videos_train.npy', 'videos_val.npy', 'videos_test.npy', 'label_map.json']. Finish the sequence-building notebook first.

In [ ]:
# Validate dimensions, labels and source-video split isolation
assert X_train.ndim == X_val.ndim == X_test.ndim == 3
assert X_train.shape[1:] == X_val.shape[1:] == X_test.shape[1:], "Sequence dimensions differ."
assert X_train.shape[2] == 51, "Expected 51 features per frame (17 joints x/y/confidence)."
assert len(X_train) == len(y_train) == len(videos_train)
assert len(X_val) == len(y_val) == len(videos_val)
assert len(X_test) == len(y_test) == len(videos_test)
assert len(X_train) and len(X_val) and len(X_test), "One of the splits is empty."

for split_name, y in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    unique, counts = np.unique(y, return_counts=True)
    found = set(unique.tolist())
    missing_names = [id_to_label[i] for i in class_ids if i not in found]
    print(f"{split_name} windows by class:", {
        id_to_label[int(k)]: int(v) for k, v in zip(unique, counts)
    })
    if missing_names:
        print("  WARNING: classes with no windows in this split:", missing_names)

train_video_set = set(np.unique(videos_train).tolist())
val_video_set = set(np.unique(videos_val).tolist())
test_video_set = set(np.unique(videos_test).tolist())
assert not train_video_set & val_video_set, "Source-video leakage between train and validation."
assert not train_video_set & test_video_set, "Source-video leakage between train and test."
assert not val_video_set & test_video_set, "Source-video leakage between validation and test."

train_counts = np.bincount(np.asarray(y_train, dtype=np.int64), minlength=NUM_CLASSES)
zero_train = [id_to_label[i] for i, n in enumerate(train_counts) if n == 0]
assert not zero_train, (
    f"No training windows for classes {zero_train}. Return to the sequence-builder audit; "
    "do not train a classifier that cannot learn these classes."
)
print("No source-video leakage detected. Sequence length:", X_train.shape[1])

## Model and batch settings

The LSTM receives 16 consecutive sampled poses (or the sequence length stored by the builder) and predicts one of the action classes. The class-weighted loss helps when some classes have more windows than others. It does not fix poor pose quality or a biased split.

In [ ]:
# Tune only if needed. Lower BATCH_SIZE if CUDA runs out of memory.
BATCH_SIZE = 128
HIDDEN_SIZE = 128
NUM_LAYERS = 1
DROPOUT = 0.35
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 80
PATIENCE = 10
NUM_WORKERS = 0

class PoseSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        # Copy one sequence at a time from the read-only memory map.
        x = np.array(self.X[idx], dtype=np.float32, copy=True)
        y = int(self.y[idx])
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.long)

train_dataset = PoseSequenceDataset(X_train, y_train)
val_dataset = PoseSequenceDataset(X_val, y_val)
test_dataset = PoseSequenceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

class PoseLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        return self.classifier(self.dropout(h_n[-1]))

model = PoseLSTM(51, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES, DROPOUT).to(DEVICE)
class_weights = len(y_train) / (NUM_CLASSES * train_counts.astype(np.float64))
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights_t)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(model)
print("Train windows:", len(train_dataset), "| batch size:", BATCH_SIZE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def run_epoch(model, loader, train_mode):
    model.train(train_mode)
    total_loss = 0.0
    total_count = 0
    truths, predictions = [], []
    context = torch.enable_grad if train_mode else torch.no_grad
    with context():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            yb = yb.to(DEVICE, non_blocking=(DEVICE.type == "cuda"))
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += float(loss.item()) * len(yb)
            total_count += len(yb)
            truths.append(yb.detach().cpu().numpy())
            predictions.append(logits.argmax(dim=1).detach().cpu().numpy())
    y_true = np.concatenate(truths) if truths else np.array([], dtype=np.int64)
    y_pred = np.concatenate(predictions) if predictions else np.array([], dtype=np.int64)
    avg_loss = total_loss / max(total_count, 1)
    acc = accuracy_score(y_true, y_pred) if total_count else 0.0
    macro_f1 = f1_score(y_true, y_pred, labels=class_ids, average="macro", zero_division=0) if total_count else 0.0
    bal_acc = balanced_accuracy_score(y_true, y_pred) if total_count else 0.0
    return avg_loss, acc, macro_f1, bal_acc

print("Training/evaluation helpers ready.")

In [ ]:
MODEL_PATH = OUTPUT_MODEL_DIR / "lstm_action_model_ucf101.pt"
HISTORY_PATH = OUTPUT_REPORT_DIR / "lstm_ucf101_training_history.csv"

history = []
best_val_f1 = -1.0
best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc, train_f1, train_bal_acc = run_epoch(model, train_loader, True)
    val_loss, val_acc, val_f1, val_bal_acc = run_epoch(model, val_loader, False)
    row = {
        "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_acc,
        "train_macro_f1": train_f1, "train_balanced_accuracy": train_bal_acc,
        "val_loss": val_loss, "val_accuracy": val_acc,
        "val_macro_f1": val_f1, "val_balanced_accuracy": val_bal_acc,
    }
    history.append(row)
    print(
        f"Epoch {epoch:02d}/{MAX_EPOCHS} | train loss={train_loss:.4f}, "
        f"acc={train_acc:.3f}, macro-F1={train_f1:.3f} | "
        f"val loss={val_loss:.4f}, acc={val_acc:.3f}, macro-F1={val_f1:.3f}",
        flush=True
    )
    improved = (val_f1 > best_val_f1 + 1e-6) or (
        abs(val_f1 - best_val_f1) <= 1e-6 and val_loss < best_val_loss
    )
    if improved:
        best_val_f1, best_val_loss, best_epoch = val_f1, val_loss, epoch
        checkpoint = {
            "model_state": copy.deepcopy(model.state_dict()),
            "input_size": 51,
            "sequence_length": int(X_train.shape[1]),
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
            "num_classes": NUM_CLASSES,
            "label_map": label_map,
            "best_epoch": best_epoch,
            "best_val_macro_f1": best_val_f1,
            "best_val_loss": best_val_loss,
        }
        torch.save(checkpoint, MODEL_PATH)
        epochs_without_improvement = 0
        print("  Saved best validation checkpoint:", MODEL_PATH.name, flush=True)
    else:
        epochs_without_improvement += 1
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping: no validation macro-F1 improvement for {PATIENCE} epochs.")
        break

assert MODEL_PATH.is_file(), "No model checkpoint was saved."
print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation macro-F1:", round(best_val_f1, 4))
print("Model:", MODEL_PATH.resolve())
print("History:", HISTORY_PATH.resolve())

In [ ]:
# Plot history
hist_df = pd.read_csv(HISTORY_PATH)
plt.figure(figsize=(8, 4))
plt.plot(hist_df["epoch"], hist_df["train_loss"], label="Train loss")
plt.plot(hist_df["epoch"], hist_df["val_loss"], label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("UCF101 LSTM loss")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
loss_plot_path = OUTPUT_REPORT_DIR / "lstm_ucf101_loss.png"
plt.savefig(loss_plot_path, dpi=160); plt.show()

plt.figure(figsize=(8, 4))
plt.plot(hist_df["epoch"], hist_df["train_macro_f1"], label="Train macro-F1")
plt.plot(hist_df["epoch"], hist_df["val_macro_f1"], label="Validation macro-F1")
plt.xlabel("Epoch"); plt.ylabel("Macro-F1"); plt.title("UCF101 LSTM macro-F1")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
f1_plot_path = OUTPUT_REPORT_DIR / "lstm_ucf101_macro_f1.png"
plt.savefig(f1_plot_path, dpi=160); plt.show()
print("Saved plots:", loss_plot_path, f1_plot_path)

In [ ]:
# Final test evaluation — run after model selection is complete
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
best_model = PoseLSTM(
    input_size=int(checkpoint["input_size"]),
    hidden_size=int(checkpoint["hidden_size"]),
    num_layers=int(checkpoint["num_layers"]),
    num_classes=int(checkpoint["num_classes"]),
    dropout=float(checkpoint["dropout"])
).to(DEVICE)
best_model.load_state_dict(checkpoint["model_state"], strict=True)
best_model.eval()

true_chunks, pred_chunks = [], []
with torch.inference_mode():
    for xb, yb in test_loader:
        logits = best_model(xb.to(DEVICE, non_blocking=(DEVICE.type == "cuda")))
        true_chunks.append(yb.numpy())
        pred_chunks.append(logits.argmax(dim=1).cpu().numpy())

y_true = np.concatenate(true_chunks).astype(np.int64)
y_pred = np.concatenate(pred_chunks).astype(np.int64)
labels_all = list(range(NUM_CLASSES))
metrics = {
    "test_accuracy": float(accuracy_score(y_true, y_pred)),
    "test_balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    "test_macro_f1": float(f1_score(y_true, y_pred, labels=labels_all, average="macro", zero_division=0)),
    "best_epoch": int(checkpoint["best_epoch"]),
    "best_validation_macro_f1": float(checkpoint["best_val_macro_f1"]),
    "test_windows": int(len(y_true)),
    "test_source_videos": int(len(test_video_set)),
    "num_classes": int(NUM_CLASSES),
}
print("Best checkpoint epoch:", metrics["best_epoch"])
print("Test accuracy:", round(metrics["test_accuracy"], 4))
print("Test balanced accuracy:", round(metrics["test_balanced_accuracy"], 4))
print("Test macro-F1:", round(metrics["test_macro_f1"], 4))
print("Test windows:", metrics["test_windows"])
print("Test source videos:", metrics["test_source_videos"])

report_text = classification_report(y_true, y_pred, labels=labels_all,
                                    target_names=[id_to_label[i] for i in labels_all],
                                    zero_division=0)
print("\nClassification report:\n", report_text)
report_dict = classification_report(y_true, y_pred, labels=labels_all,
                                    target_names=[id_to_label[i] for i in labels_all],
                                    output_dict=True, zero_division=0)
report_csv = OUTPUT_REPORT_DIR / "lstm_ucf101_classification_report.csv"
pd.DataFrame(report_dict).transpose().to_csv(report_csv)
metrics_path = OUTPUT_REPORT_DIR / "lstm_ucf101_test_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

cm = confusion_matrix(y_true, y_pred, labels=labels_all)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(max(12, NUM_CLASSES * 0.22), max(10, NUM_CLASSES * 0.20)))
cm_display.plot(ax=ax, xticks_rotation=90, colorbar=False, values_format="d")
ax.set_title("UCF101 LSTM — held-out test confusion matrix")
plt.tight_layout()
cm_path = OUTPUT_REPORT_DIR / "lstm_ucf101_confusion_matrix.png"
plt.savefig(cm_path, dpi=160)
plt.show()

# Correct/predicted count table without shadowing IPython's display function.
compare_df = pd.DataFrame({
    "true_label": [id_to_label[int(i)] for i in y_true],
    "predicted_label": [id_to_label[int(i)] for i in y_pred],
    "correct": y_true == y_pred,
})
print(compare_df.head(20).to_string(index=False))
print("\nSaved report:", report_csv.resolve())
print("Saved metrics:", metrics_path.resolve())
print("Saved confusion matrix:", cm_path.resolve())

## Interpreting the result

- Use test accuracy together with **macro-F1** and balanced accuracy, since UCF101 classes may be imbalanced in the processed set.
- Do not compare results fairly if the source-video split changed between experiments.
- This is pose-based recognition, so performance depends on person detection and keypoint quality. It does not use RGB appearance features.
- The test set is held out by source video, but this does not automatically guarantee independence between very similar recordings; the extraction notebook attempts to diversify source recording groups only during optional per-class pilot selection.
- If the kernel runs out of memory, reduce `BATCH_SIZE` from 128 to 64 or 32. Feature arrays are memory-mapped, so they are not loaded fully into RAM.